# Z-Split Normalization (Zero-Preserving Split Normalization)

**Author:** Abdulrhman Almoadi  
**Affiliation:** King Abdulaziz City for Science and Technology (KACST)  
**Repository:** [github.com/aalmoadi/endwi](https://github.com/aalmoadi/endwi)  

---

## What is Z-Split?

Z-Split is a normalization technique developed for bipolar spectral indices where **zero carries physical meaning as a class boundary** (e.g., water vs. non-water in ENDWI).

### Formula

$$
Z_{\text{split}}(x) =
\begin{cases}
\dfrac{x}{\max(X^+)} & \text{if } x > 0 \\
0 & \text{if } x = 0 \\
\dfrac{x}{|\min(X^-)|} & \text{if } x < 0
\end{cases}
$$

Where:
- $X^+$ = all positive values in the array
- $X^-$ = all negative values in the array
- Output range: $[-1, 1]$ with zero strictly preserved

### Why Z-Split for ENDWI?

ENDWI = NDWI / Green band. When computed from DN values, the raw range is extremely narrow (~−0.0004 to ~0.00015), making Otsu thresholding unstable. Z-Split expands this range to [−1, 1] while preserving zero as the natural water/non-water boundary.

---

## Citation

If you use Z-Split, please cite:

> Almoadi, A.: Reducing False Alarms in Urban Flood Detection: An Enhanced NDWI (ENDWI) with Hybrid Max Fusion on Sentinel-2 Data, EGUsphere [preprint], https://doi.org/10.5194/egusphere-2026-672, 2026.

## 1. Import Libraries

In [ ]:
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

print('Libraries loaded successfully.')

## 2. Z-Split Core Function

In [ ]:
def zsplit_normalize(array, nodata=None):
    """
    Apply Z-Split (Zero-Preserving Split Normalization) to a NumPy array.

    Positive values are normalized to [0, 1] by dividing by max(positive values).
    Negative values are normalized to [-1, 0] by dividing by abs(min(negative values)).
    Zero values remain exactly zero.

    Parameters
    ----------
    array : np.ndarray
        Input 2D array (e.g., raw ENDWI raster).
    nodata : float or None
        NoData value to exclude from normalization. These pixels are set to NaN.

    Returns
    -------
    result : np.ndarray
        Normalized array in range [-1, 1] with zero preserved.
    """
    arr = array.astype(np.float64).copy()

    # Mask nodata
    if nodata is not None:
        arr[arr == nodata] = np.nan

    result = np.zeros_like(arr)

    # Positive half: normalize to [0, 1]
    pos_mask = arr > 0
    if pos_mask.any():
        max_pos = np.nanmax(arr[pos_mask])
        if max_pos != 0:
            result[pos_mask] = arr[pos_mask] / max_pos

    # Negative half: normalize to [-1, 0]
    neg_mask = arr < 0
    if neg_mask.any():
        min_neg = np.nanmin(arr[neg_mask])  # most negative value
        if min_neg != 0:
            result[neg_mask] = arr[neg_mask] / abs(min_neg)

    # Preserve NaN for nodata
    result[np.isnan(arr)] = np.nan

    return result

print('zsplit_normalize() defined.')

## 3. Apply Z-Split to ENDWI Raster

Update the path below to point to your ENDWI raw raster.

In [ ]:
# ── USER INPUT ──────────────────────────────────────────────────────────────
endwi_raw_path = r'/path/to/your/ENDWI_raw.tif'
output_path    = r'/path/to/your/ENDWI_zsplit.tif'
# ────────────────────────────────────────────────────────────────────────────

with rasterio.open(endwi_raw_path) as src:
    raw = src.read(1).astype(np.float64)
    profile = src.profile
    nodata_val = src.nodata

print(f'Raw ENDWI loaded: shape={raw.shape}')
print(f'Raw value range: min={np.nanmin(raw):.8f}, max={np.nanmax(raw):.8f}')
print(f'NoData value: {nodata_val}')

In [ ]:
# Apply Z-Split
endwi_zsplit = zsplit_normalize(raw, nodata=nodata_val)

print(f'Z-Split ENDWI range: min={np.nanmin(endwi_zsplit):.4f}, max={np.nanmax(endwi_zsplit):.4f}')
print(f'Zero preserved: {np.sum(raw == 0)} pixels with value 0 in raw → {np.sum(endwi_zsplit == 0)} pixels with value 0 after Z-Split')

In [ ]:
# Save Z-Split raster
profile.update(dtype=rasterio.float32, nodata=np.nan)

with rasterio.open(output_path, 'w', **profile) as dst:
    dst.write(endwi_zsplit.astype(np.float32), 1)

print(f'Saved: {output_path}')

## 4. Visualization: Before vs. After Z-Split

In [ ]:
fig = plt.figure(figsize=(14, 5))
gs = gridspec.GridSpec(1, 2, figure=fig)

raw_flat = raw[~np.isnan(raw)].flatten()
zsplit_flat = endwi_zsplit[~np.isnan(endwi_zsplit)].flatten()

# --- Before Z-Split ---
ax1 = fig.add_subplot(gs[0])
ax1.hist(raw_flat, bins=100, color='steelblue', edgecolor='none', alpha=0.85)
ax1.set_title(
    f'ENDWI raw.tif — Before Z-Split\n(Narrow Range: {np.nanmin(raw):.4f} to {np.nanmax(raw):.4f})',
    fontsize=10
)
ax1.set_xlabel('Pixel Value', fontsize=9)
ax1.set_ylabel('Frequency', fontsize=9)
ax1.axvline(0, color='red', linewidth=1.2, linestyle='--', label='Zero boundary')
ax1.legend(fontsize=8)

# --- After Z-Split ---
ax2 = fig.add_subplot(gs[1])
neg_vals = zsplit_flat[zsplit_flat < 0]
pos_vals = zsplit_flat[zsplit_flat > 0]
ax2.hist(neg_vals, bins=60, color='steelblue', edgecolor='none', alpha=0.85, label='Non-water (negative)')
ax2.hist(pos_vals, bins=60, color='salmon',    edgecolor='none', alpha=0.85, label='Water (positive)')
ax2.set_title(
    f'ENDWI raw.tif — After Z-Split Normalization\n(Expanded Range: [-1, 1])',
    fontsize=10
)
ax2.set_xlabel('Pixel Value', fontsize=9)
ax2.set_ylabel('Frequency', fontsize=9)
ax2.axvline(0, color='red', linewidth=1.2, linestyle='--', label='Zero boundary')
ax2.legend(fontsize=8)

plt.suptitle('Z-Split Normalization — ENDWI Histogram Comparison', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('zsplit_histogram_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: zsplit_histogram_comparison.png')

## 5. Comparison: Z-Split vs. Min-Max vs. Z-Score

In [ ]:
def minmax_normalize(array, nodata=None):
    arr = array.astype(np.float64).copy()
    if nodata is not None:
        arr[arr == nodata] = np.nan
    mn, mx = np.nanmin(arr), np.nanmax(arr)
    return (arr - mn) / (mx - mn) if mx != mn else np.zeros_like(arr)

def zscore_normalize(array, nodata=None):
    arr = array.astype(np.float64).copy()
    if nodata is not None:
        arr[arr == nodata] = np.nan
    mu, sigma = np.nanmean(arr), np.nanstd(arr)
    return (arr - mu) / sigma if sigma != 0 else np.zeros_like(arr)

endwi_minmax = minmax_normalize(raw, nodata=nodata_val)
endwi_zscore = zscore_normalize(raw, nodata=nodata_val)

print('Normalization comparison:')
print(f'  Z-Split  → range: [{np.nanmin(endwi_zsplit):.4f}, {np.nanmax(endwi_zsplit):.4f}]  | zero preserved: YES')
print(f'  Min-Max  → range: [{np.nanmin(endwi_minmax):.4f}, {np.nanmax(endwi_minmax):.4f}]  | zero preserved: NO (shifted)')
print(f'  Z-Score  → range: [{np.nanmin(endwi_zscore):.4f}, {np.nanmax(endwi_zscore):.4f}]  | zero preserved: NO (centered on mean)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

methods = [
    (endwi_zsplit,  'Z-Split (Proposed)',  'salmon'),
    (endwi_minmax,  'Min-Max',             'steelblue'),
    (endwi_zscore,  'Z-Score',             'mediumpurple'),
]

for ax, (data, title, color) in zip(axes, methods):
    flat = data[~np.isnan(data)].flatten()
    ax.hist(flat, bins=80, color=color, edgecolor='none', alpha=0.85)
    ax.axvline(0, color='red', linewidth=1.2, linestyle='--')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Normalized Value', fontsize=9)
    ax.set_ylabel('Frequency', fontsize=9)

plt.suptitle('ENDWI Normalization Methods Comparison', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('zsplit_methods_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: zsplit_methods_comparison.png')

## 6. Quick Demo with Synthetic Data

No raster file needed — demonstrates Z-Split behaviour on simulated ENDWI-like values.

In [ ]:
np.random.seed(42)

# Simulate ENDWI-like distribution (narrow near-zero range, DN scale)
non_water = np.random.normal(-0.00015, 0.00008, 7000)   # non-flooded pixels
water     = np.random.normal( 0.00008, 0.00003, 3000)   # flooded pixels
demo_array = np.concatenate([non_water, water]).reshape(100, 100)

demo_zsplit = zsplit_normalize(demo_array)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(demo_array.flatten(), bins=80, color='steelblue', alpha=0.85)
axes[0].axvline(0, color='red', linewidth=1.5, linestyle='--', label='Zero boundary')
axes[0].set_title(f'Synthetic ENDWI — Raw\nRange: [{demo_array.min():.5f}, {demo_array.max():.5f}]', fontsize=10)
axes[0].set_xlabel('Value'); axes[0].set_ylabel('Frequency')
axes[0].legend(fontsize=8)

neg = demo_zsplit[demo_zsplit < 0].flatten()
pos = demo_zsplit[demo_zsplit > 0].flatten()
axes[1].hist(neg, bins=60, color='steelblue', alpha=0.85, label='Non-water')
axes[1].hist(pos, bins=60, color='salmon',    alpha=0.85, label='Water')
axes[1].axvline(0, color='red', linewidth=1.5, linestyle='--', label='Zero boundary')
axes[1].set_title(f'Synthetic ENDWI — After Z-Split\nRange: [{demo_zsplit.min():.4f}, {demo_zsplit.max():.4f}]', fontsize=10)
axes[1].set_xlabel('Normalized Value'); axes[1].set_ylabel('Frequency')
axes[1].legend(fontsize=8)

plt.suptitle('Z-Split Demo — Synthetic ENDWI Data', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('zsplit_demo_synthetic.png', dpi=150, bbox_inches='tight')
plt.show()
print('Demo complete. Zero preserved:', np.sum(demo_zsplit == 0), 'pixels.')